# 04 - Deep hedging sous shifted Bergomi

Objectif: reprendre le pipeline deep hedging sur Bergomi, s?par? du cas Heston.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
from torch.utils.data import DataLoader, TensorDataset

from deep_hedging import VNN, evaluate, get_optimizer, training_loop
from models import compute_payoff_europeen, simulate_shifted_bergomi


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
I_train, I_test, J = 1_024, 512, 40
T, V0, K = 1/12, 0.0625, 1.0

S_train, dS_train, sigma_train, states_train, W_hat_train = simulate_shifted_bergomi(1, 1.6, V0, 1/52, 0.1, -0.7, T, J, I_train, seed=3)
S_test, dS_test, sigma_test, states_test, W_hat_test = simulate_shifted_bergomi(1, 1.6, V0, 1/52, 0.1, -0.7, T, J, I_test, seed=4)

payoff_train = compute_payoff_europeen(S_train, K)
payoff_test = compute_payoff_europeen(S_test, K)


In [ ]:
train_loader = DataLoader(TensorDataset(
    torch.tensor(states_train, dtype=torch.float32),
    torch.tensor(dS_train, dtype=torch.float32),
    torch.tensor(payoff_train, dtype=torch.float32),
), batch_size=64, shuffle=True)

test_loader = DataLoader(TensorDataset(
    torch.tensor(states_test, dtype=torch.float32),
    torch.tensor(dS_test, dtype=torch.float32),
    torch.tensor(payoff_test, dtype=torch.float32),
), batch_size=256)


In [ ]:
model = VNN(input_features=3, output_features=1).to(device)
optimizer, scheduler = get_optimizer(model)
training_loop(model, train_loader, 64, optimizer, scheduler, num_epochs=2, device=device)
hedging_pnl, mse, mean_pnl = evaluate(model, test_loader, device)
print('Bergomi VNN X0:', float(model.X0.detach().cpu()))
print('Bergomi VNN MSE:', float(mse.detach().cpu()), 'Mean PnL:', float(mean_pnl.detach().cpu()))
